# 🚗 Simulating Self-Driving Cars
### Deep Learning + Computer Vision | GSSoC 2026 | Issue #383

**Two core capabilities:**
1. 🛣️ **Lane Detection** — Hough Transform + weighted lane averaging
2. 🚦 **Traffic Regulation Detection** — YOLOv8 real-time object detection
3. 🎯 **Steering Angle Prediction** — Geometric lane-center tracking
4. 🎮 **2D Top-Down Simulation** — matplotlib animated car navigation

| Library | Purpose |
|---|---|
| `OpenCV` | Image processing & edge detection |
| `YOLOv8` (ultralytics) | Traffic sign & vehicle detection |
| `NumPy` | Numerical operations |
| `Matplotlib` | Visualization & simulation |

> **Author:** GSSoC 2026 Contributor | **Issue:** [#383](https://github.com/Niketkumardheeryan/ML-CaPsule/issues/383)


In [ ]:
# Install dependencies (run once)
!pip install -q opencv-python-headless ultralytics matplotlib numpy Pillow
print("✅ All dependencies installed!")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100
print(f"✅ Imports ready | OpenCV {cv2.__version__}")


## 📸 Sample Road Image
We generate a **synthetic road scene** for demonstration. Replace `sample_image` with any real road image or video frame.


In [ ]:
def create_road_image(w=1280, h=720):
    """Generate a synthetic road scene with lane markings and a stop sign."""
    img = np.zeros((h, w, 3), dtype=np.uint8)

    # Sky gradient
    for y in range(h // 2):
        r = y / (h // 2)
        img[y] = [int(235*(1-r*0.05)), int(206*(1-r*0.1)), int(135*(1-r*0.3))]

    # Road surface
    road_pts = np.array([[int(w*0.25), h//2],[int(w*0.75), h//2],[w,h],[0,h]], np.int32)
    cv2.fillPoly(img, [road_pts], (55, 55, 55))

    # White lane lines
    cv2.line(img, (int(w*0.38), int(h*0.55)), (int(w*0.03), h-5), (255,255,255), 9)
    cv2.line(img, (int(w*0.62), int(h*0.55)), (int(w*0.97), h-5), (255,255,255), 9)

    # Yellow center dashes
    for i in range(8):
        ts, te = i/8, (i+0.45)/8
        def lerp(a,b,t): return int(a+(b-a)*t)
        cv2.line(img,
            (lerp(w//2,w//2,ts), lerp(int(h*0.55),h-5,ts)),
            (lerp(w//2,w//2,te), lerp(int(h*0.55),h-5,te)),
            (0,220,220), 5)

    # Stop sign
    cv2.rectangle(img, (870,270),(930,330),(0,0,220),-1)
    cv2.rectangle(img, (870,270),(930,330),(255,255,255),3)
    cv2.putText(img,"STOP",(877,308),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)
    cv2.line(img,(900,330),(900,390),(120,80,40),5)

    # Trees
    for tx,ty in [(60,290),(1210,290),(130,340),(1150,340)]:
        cv2.circle(img,(tx,ty),32,(0,110,0),-1)
        cv2.rectangle(img,(tx-5,ty+28),(tx+5,ty+65),(90,55,20),-1)

    return img

sample_image = create_road_image()
plt.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
plt.title("🛣️ Synthetic Road Scene", fontsize=15, fontweight='bold')
plt.axis('off'); plt.tight_layout()
plt.savefig('sample_road.png', bbox_inches='tight')
plt.show()
print("✅ Road image created!")


## 🛣️ Section 1 — Lane Detection

### Pipeline
```
Original → Grayscale → Gaussian Blur → Canny Edges → ROI Mask → Hough Lines → Averaged Lanes
```


In [ ]:
# ── LANE DETECTION MODULE ──────────────────────────────────────────────────────

def to_gray(img):        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
def blur(img, k=5):      return cv2.GaussianBlur(img, (k,k), 0)
def edges(img, lo=50, hi=150): return cv2.Canny(img, lo, hi)

def roi(img):
    """Keep only the trapezoidal road region."""
    h, w = img.shape[:2]
    pts = np.array([[(int(w*.05),h),(int(w*.42),int(h*.58)),
                     (int(w*.58),int(h*.58)),(int(w*.95),h)]], np.int32)
    mask = np.zeros_like(img)
    cv2.fillPoly(mask, pts, 255 if img.ndim==2 else (255,255,255))
    return cv2.bitwise_and(img, mask)

def weighted_line(fits, weights):
    """Weighted average of (slope, intercept) pairs."""
    if not fits: return None
    total = sum(weights)
    s = sum(f[0]*w for f,w in zip(fits,weights)) / total
    b = sum(f[1]*w for f,w in zip(fits,weights)) / total
    return s, b

def fit_lanes(lines):
    """Separate detected segments into left/right lanes and average them."""
    L, R, Lw, Rw = [], [], [], []
    if lines is None: return None, None
    for l in lines:
        for x1,y1,x2,y2 in l:
            if x2==x1: continue
            sl = (y2-y1)/(x2-x1)
            ic = y1 - sl*x1
            ln = np.hypot(x2-x1, y2-y1)
            if -0.9<sl<-0.35: L.append((sl,ic)); Lw.append(ln)
            elif 0.35<sl<0.9: R.append((sl,ic)); Rw.append(ln)
    return weighted_line(L,Lw), weighted_line(R,Rw)

def line_pts(h, params, y0=1.0, y1=0.58):
    """Convert (slope,intercept) to pixel endpoints."""
    if params is None: return None
    s, b = params
    if abs(s)<1e-6: return None
    Y0,Y1 = int(h*y0), int(h*y1)
    return (int((Y0-b)/s), Y0), (int((Y1-b)/s), Y1)

def draw_lanes(img, left, right):
    """Draw lane lines + filled region on image."""
    ov = np.zeros_like(img)
    if left:  cv2.line(ov, left[0],  left[1],  (0,230,0), 10)
    if right: cv2.line(ov, right[0], right[1], (0,230,0), 10)
    if left and right:
        pts = np.array([left[0],left[1],right[1],right[0]], np.int32)
        cv2.fillPoly(ov, [pts], (0,100,255))
    return cv2.addWeighted(img, 0.82, ov, 0.5, 0)

def steer_angle(img, left, right):
    """Compute steering angle (degrees) from lane-center offset."""
    h, w = img.shape[:2]
    cx = w//2
    if left is None and right is None: return 0.0
    if left and right: lane_x = (left[0][0]+right[0][0])//2
    elif left:         lane_x = left[0][0] + int(w*0.25)
    else:              lane_x = right[0][0] - int(w*0.25)
    return round(np.degrees(np.arctan2(cx-lane_x, h*0.5)), 2)

def lane_pipeline(img):
    """Run full lane detection. Returns (annotated_img, left, right, angle)."""
    g = to_gray(img); bl = blur(g); ed = edges(bl); ro = roi(ed)
    hlines = cv2.HoughLinesP(ro, 2, np.pi/180, 50, minLineLength=40, maxLineGap=150)
    lp, rp = fit_lanes(hlines)
    h = img.shape[0]
    left, right = line_pts(h,lp), line_pts(h,rp)
    angle = steer_angle(img, left, right)
    return draw_lanes(img.copy(), left, right), left, right, angle

print("✅ Lane Detection module ready!")


In [ ]:
# ── VISUALISE ALL PIPELINE STEPS ───────────────────────────────────────────────

img = sample_image.copy()
g  = to_gray(img)
bl = blur(g)
ed = edges(bl)
ro = roi(ed)
result, left, right, angle = lane_pipeline(img)

fig, axes = plt.subplots(2, 3, figsize=(18,10))
fig.suptitle("🛣️ Lane Detection — Step by Step", fontsize=18, fontweight='bold')

steps = [
    (cv2.cvtColor(img, cv2.COLOR_BGR2RGB), "1. Original"),
    (g,  "2. Grayscale"),
    (bl, "3. Gaussian Blur"),
    (ed, "4. Canny Edges"),
    (ro, "5. ROI Mask"),
    (cv2.cvtColor(result, cv2.COLOR_BGR2RGB), f"6. Lanes Detected ✅  |  Steering: {angle}°"),
]
for ax, (im, title) in zip(axes.flatten(), steps):
    cmap = 'gray' if im.ndim == 2 else None
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=12, fontweight='bold'); ax.axis('off')

plt.tight_layout()
plt.savefig('lane_pipeline_steps.png', bbox_inches='tight')
plt.show()
print(f"Left: {'✅' if left else '❌'}  |  Right: {'✅' if right else '❌'}  |  Angle: {angle}°")


## 🚦 Section 2 — Traffic & Object Detection (YOLOv8)

**YOLOv8 Nano** detects road-relevant objects in real time:

| Class | Label |
|---|---|
| 0 | 🚶 Person |
| 2 | 🚗 Car |
| 5 | 🚌 Bus |
| 7 | 🚚 Truck |
| 9 | 🚦 Traffic Light |
| 11 | 🛑 Stop Sign |


In [ ]:
from ultralytics import YOLO

TRAFFIC_CLASSES = {0:'🚶 Person',1:'🚲 Bicycle',2:'🚗 Car',3:'🛵 Motorcycle',
                   5:'🚌 Bus',7:'🚚 Truck',9:'🚦 Traffic Light',11:'🛑 Stop Sign'}
CLASS_COLORS    = {0:(0,255,255),1:(255,165,0),2:(0,255,0),3:(255,0,255),
                   5:(255,0,0),7:(0,0,255),9:(0,215,255),11:(0,0,180)}

print("📥 Loading YOLOv8 Nano (~6 MB)…")
yolo = YOLO('yolov8n.pt')
print("✅ YOLOv8 Nano ready!")


In [ ]:
def detect_objects(img, model, conf=0.25):
    """Run YOLOv8 detection and draw labelled bounding boxes."""
    out = img.copy(); dets = []
    for res in model(img, conf=conf, verbose=False):
        for box in res.boxes:
            cid = int(box.cls[0]); conf_v = float(box.conf[0])
            if cid not in TRAFFIC_CLASSES: continue
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            col = CLASS_COLORS.get(cid,(255,255,255))
            label = f"{TRAFFIC_CLASSES[cid]} {conf_v:.0%}"
            cv2.rectangle(out,(x1,y1),(x2,y2),col,3)
            (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.6,2)
            cv2.rectangle(out,(x1,y1-th-10),(x1+tw+6,y1),col,-1)
            cv2.putText(out,label,(x1+3,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,0),2)
            dets.append({'class':TRAFFIC_CLASSES[cid],'conf':conf_v,'bbox':(x1,y1,x2,y2)})
    return out, dets

det_img, dets = detect_objects(sample_image, yolo)

fig, axes = plt.subplots(1,2,figsize=(18,7))
axes[0].imshow(cv2.cvtColor(sample_image,cv2.COLOR_BGR2RGB))
axes[0].set_title("Original",fontsize=14,fontweight='bold'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(det_img,cv2.COLOR_BGR2RGB))
axes[1].set_title(f"🚦 YOLOv8 Detection — {len(dets)} object(s) found",fontsize=14,fontweight='bold'); axes[1].axis('off')
plt.tight_layout()
plt.savefig('object_detection.png', bbox_inches='tight')
plt.show()
print("Detections:", [d['class'] for d in dets] or ["None in synthetic image — works on real road footage!"])


## 🤖 Section 3 — Full Self-Driving Pipeline

Combines **lane detection + object detection + steering** into one pass.


In [ ]:
def full_pipeline(img, model):
    """
    Complete self-driving perception pipeline.
    1. Lane detection + steering angle
    2. Traffic/object detection
    3. HUD overlay (steering, speed, detections)
    Returns annotated image.
    """
    # Step 1: Lanes
    lane_img, left, right, angle = lane_pipeline(img)

    # Step 2: Objects
    final, dets = detect_objects(lane_img, model)

    # Step 3: HUD
    h, w = final.shape[:2]
    direction = "◀ LEFT" if angle>2 else "▶ RIGHT" if angle<-2 else "▲ STRAIGHT"

    cv2.rectangle(final,(10,10),(320,130),(0,0,0),-1)
    cv2.rectangle(final,(10,10),(320,130),(0,200,0),2)
    cv2.putText(final,f"Steering : {angle:.1f}°",(20,50),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,230,0),2)
    cv2.putText(final,f"Direction: {direction}",(20,85),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,220,0),2)
    cv2.putText(final,f"Objects  : {len(dets)}",(20,120),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,200,255),2)

    # Steering arrow
    cx,cy = w//2, int(h*0.87)
    ex = int(cx - 90*np.sin(np.radians(angle)))
    ey = int(cy - 90*np.cos(np.radians(angle)))
    cv2.arrowedLine(final,(cx,cy),(ex,ey),(255,140,0),4,tipLength=0.3)

    return final

result_full = full_pipeline(sample_image.copy(), yolo)
plt.figure(figsize=(14,7))
plt.imshow(cv2.cvtColor(result_full,cv2.COLOR_BGR2RGB))
plt.title("🤖 Full Self-Driving Pipeline Output",fontsize=16,fontweight='bold')
plt.axis('off'); plt.tight_layout()
plt.savefig('full_pipeline_result.png', bbox_inches='tight')
plt.show()
print("✅ Full pipeline complete!")


## 🎮 Section 4 — 2D Top-Down Car Simulation

A matplotlib animation of a car navigating a **sinusoidal curved road**, using the steering controller to stay centred.


In [ ]:
W, H = 600, 700
ROAD_W = 110
CAR_W, CAR_H = 28, 48

class CarSimulator:
    def __init__(self):
        self.x = W//2; self.y = H-100
        self.offset = 0; self.trail = []; self.steers = []

    def road_cx(self, y, off=0):
        return int(W//2 + 140*np.sin((y+off)*0.009))

    def step(self):
        target = self.road_cx(self.y-90, self.offset)
        steer  = np.clip((self.x-target)*0.45, -28, 28)
        self.x = int(np.clip(self.x - steer*0.12, 60, W-60))
        self.offset += 3
        self.trail.append((self.x, self.y))
        self.steers.append(steer)
        if len(self.trail)>55: self.trail.pop(0)
        return steer

fig, (ax_sim, ax_steer) = plt.subplots(1,2, figsize=(13,8))
fig.patch.set_facecolor('#0d0d1a')
sim = CarSimulator()

def draw_frame(frame):
    steer = sim.step()

    # ── Simulation panel ──────────────────────────────────────
    ax_sim.clear(); ax_sim.set_xlim(0,W); ax_sim.set_ylim(0,H)
    ax_sim.set_facecolor('#0d1117'); ax_sim.axis('off')
    ax_sim.set_title("🚗 Self-Driving Simulation", color='white', fontsize=13, fontweight='bold')

    # Road
    for y in range(0,H,3):
        cx = sim.road_cx(y, sim.offset)
        ax_sim.plot([cx-ROAD_W,cx+ROAD_W],[y,y],color='#3a3a3a',lw=3)

    # Lane lines
    lane_ys = np.arange(0, H, 1)
    lx = [sim.road_cx(y,sim.offset)-ROAD_W for y in lane_ys]
    rx = [sim.road_cx(y,sim.offset)+ROAD_W for y in lane_ys]
    ax_sim.plot(lx, lane_ys, color='white', lw=2)
    ax_sim.plot(rx, lane_ys, color='white', lw=2)

    # Center dashes
    for y in range(0,H,30):
        cx = sim.road_cx(y,sim.offset)
        ax_sim.plot([cx,cx],[y,y+15],color='#ffdd00',lw=2)

    # Trail
    if len(sim.trail)>1:
        tx,ty = zip(*sim.trail)
        for i in range(len(tx)-1):
            alpha = (i+1)/len(tx)
            ax_sim.plot([tx[i],tx[i+1]],[ty[i],ty[i+1]],color=(1,0.4,0,alpha),lw=2)

    # Car body
    car = plt.Rectangle((sim.x-CAR_W//2, sim.y-CAR_H//2), CAR_W, CAR_H,
                         color='#00aaff', zorder=5)
    ax_sim.add_patch(car)
    ax_sim.plot(sim.x, sim.y+CAR_H//2+6, 's', color='#ff4444', ms=7)  # brake light

    # HUD
    dir_txt = "◀ LEFT" if steer>2 else "▶ RIGHT" if steer<-2 else "▲ STRAIGHT"
    ax_sim.text(10,H-20, f"Steering: {steer:.1f}°  {dir_txt}",
                color='#00ff99', fontsize=11, fontweight='bold')
    ax_sim.text(10,H-45, f"Frame: {frame}",
                color='#aaaaaa', fontsize=9)

    # ── Steering chart ────────────────────────────────────────
    ax_steer.clear()
    ax_steer.set_facecolor('#0d1117')
    ax_steer.set_title("Steering Angle Over Time", color='white', fontsize=12)
    if len(sim.steers)>1:
        xs = range(len(sim.steers))
        ax_steer.plot(xs, sim.steers, color='#00ff99', lw=1.5)
        ax_steer.fill_between(xs, sim.steers, alpha=0.25, color='#00ff99')
    ax_steer.axhline(0, color='#555', lw=1, ls='--')
    ax_steer.set_ylabel("Angle (°)", color='white')
    ax_steer.set_xlabel("Frame",    color='white')
    ax_steer.tick_params(colors='white')
    for sp in ax_steer.spines.values(): sp.set_color('#444')

ani = animation.FuncAnimation(fig, draw_frame, frames=120, interval=40, blit=False)
plt.tight_layout()
# Save as gif (requires pillow) or display inline
try:
    ani.save('simulation.gif', writer='pillow', fps=25)
    print("✅ Simulation saved as simulation.gif")
except Exception as e:
    print(f"GIF save note: {e}")

HTML(ani.to_jshtml())


## 📊 Section 5 — Results & Summary

| Module | Method | Output |
|---|---|---|
| Lane Detection | Hough Transform + weighted averaging | Lane lines, steering angle |
| Object Detection | YOLOv8 Nano (pre-trained COCO) | Bounding boxes + labels |
| Steering Control | Geometric lane-center offset | Degrees left/right |
| Simulation | Matplotlib animation | Real-time 2D navigation |

### 🔮 Future Improvements
- Replace Hough-based lanes with a **U-Net semantic segmentation** model
- Add **PID controller** for smoother steering
- Integrate **GPS waypoint following**
- Use **CARLA simulator** for 3D testing
- Train custom YOLOv8 on traffic-sign datasets (GTSRB, Mapillary)


In [ ]:
# Final summary figure
fig, axes = plt.subplots(1,3,figsize=(18,6))
fig.suptitle("🚗 Self-Driving Car Simulation — Results Summary", fontsize=16, fontweight='bold')

for ax, (fname, title) in zip(axes, [
    ('lane_pipeline_steps.png', '🛣️ Lane Detection'),
    ('object_detection.png',    '🚦 Object Detection'),
    ('full_pipeline_result.png','🤖 Full Pipeline'),
]):
    try:
        from PIL import Image as PILImage
        im = np.array(PILImage.open(fname))
        ax.imshow(im)
    except:
        ax.text(0.5,0.5,"Run previous cells first",ha='center',va='center',transform=ax.transAxes)
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')

plt.tight_layout()
plt.savefig('results_summary.png', bbox_inches='tight')
plt.show()
print("✅ Project complete — ready for PR submission!")
